### 1. Install Dependencies

In [1]:
!python -m pip install -q timm transformers accelerate pandas matplotlib scikit-learn wandb kaggle


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


### 2. Imports

In [2]:
import os
import re
import json
import math
import shutil
import zipfile
import subprocess
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import wandb
import timm

from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
from transformers import VideoMAEConfig, VideoMAEForVideoClassification

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)


[aiter] start build [module_aiter_enum] under /workspace/aiter/aiter/jit/build/module_aiter_enum
Successfully preprocessed all matching files.
[aiter] finish build [module_aiter_enum], cost 21.98416202s
/opt/venv/lib/python3.10/site-packages/apex/transformer/functional/fused_rope.py:49: UserWarning: Aiter backend is selected for fused RoPE. This has lower precision. To disable aiter, export USE_ROCM_AITER_ROPE_BACKEND=0
  warnings.warn("Aiter backend is selected for fused RoPE. This has lower precision. To disable aiter, export USE_ROCM_AITER_ROPE_BACKEND=0", UserWarning)
/opt/venv/lib/python3.10/site-packages/torchao/utils.py:408: UserWarning: TORCH_VERSION_AT_LEAST_2_5 is deprecated and will be removed in torchao 0.14.0
  warnings.warn(self.msg)


### 3. Global Configuration

In [3]:
SEED = 42

WORKSPACE_ROOT = Path("/shared-docker").resolve()
RUNTIME_ROOT = WORKSPACE_ROOT / "wdf46f_runtime"

DATASET_ROOT = RUNTIME_ROOT / "wilddeepfake-46f"
INDEX_ROOT = WORKSPACE_ROOT

KAGGLE_ZIP_DIR = RUNTIME_ROOT / "kaggle_zip"
MODEL_ROOT = RUNTIME_ROOT / "kaggle_models"
MODEL_CHECKPOINT = MODEL_ROOT / "videomae-base-finetuned-ssv2"

KAGGLE_DATASET_SLUG = "afenmarbun/wilddeepfake-46f"
KAGGLE_MODEL_VERSION_SLUG = "afenmarbun/videomae-base-finetuned-ssv2/pytorch/default/1"

INFERENCE_MANIFEST_PATH = WORKSPACE_ROOT / "inference_manifest.csv"
WANDB_ARTIFACT_DOWNLOAD_ROOT = WORKSPACE_ROOT / "wandb_artifacts"
INFERENCE_OUTPUT_ROOT = WORKSPACE_ROOT / "inference_outputs"

WANDB_ENTITY = "afenmarbun-institut-teknologi-sumatera"
WANDB_PROJECT = "Tugas Akhir"
WANDB_GROUP = "Inference"
WANDB_JOB_TYPE = "inference"
WANDB_MODE = "online"

AUTO_DOWNLOAD_DATASET = True
AUTO_DOWNLOAD_VIDEOMAE_BASE_MODEL = True
FORCE_REDOWNLOAD_DATASET = False
FORCE_REEXTRACT_DATASET = False
FORCE_REDOWNLOAD_VIDEOMAE_BASE_MODEL = False

BATCH_SIZE = 32
NUM_WORKERS = 8

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.float16

CLASS_TO_IDX = {"real": 0, "fake": 1}
IDX_TO_CLASS = {0: "real", 1: "fake"}
POSITIVE_CLASS_INDEX = 1
POSITIVE_CLASS_NAME = "fake"

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

NORM_MEAN = (0.485, 0.456, 0.406)
NORM_STD = (0.229, 0.224, 0.225)

SPATIAL_MODEL_NAME = "vit_base_patch16_224.mae"
VIDEOMAE_MODEL_NAME = "videomae-base-finetuned-ssv2"

WANDB_ARTIFACT_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
INFERENCE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("DEVICE:", DEVICE)
print("WORKSPACE_ROOT:", WORKSPACE_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("INDEX_ROOT:", INDEX_ROOT)
print("INFERENCE_MANIFEST_PATH:", INFERENCE_MANIFEST_PATH)


DEVICE: cuda
WORKSPACE_ROOT: /shared-docker
DATASET_ROOT: /shared-docker/wdf46f_runtime/wilddeepfake-46f
INDEX_ROOT: /shared-docker
INFERENCE_MANIFEST_PATH: /shared-docker/inference_manifest.csv


### 4. Reproducibility & File Utilities

In [4]:
def seed_everything(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    torch.use_deterministic_algorithms(True, warn_only=True)

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False


seed_everything(SEED)


def print_section(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def print_kv(key: str, value):
    print(f"{key:32s}: {value}")


def run_command(cmd: list[str]):
    print("Menjalankan:", " ".join(cmd))

    result = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if result.stdout:
        print("\nSTDOUT:")
        print(result.stdout)

    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            "Perintah gagal dijalankan.\n"
            f"Return code: {result.returncode}\n"
            f"Command: {' '.join(cmd)}"
        )


def has_directory_content(path: Path, ignored_names: set[str] | None = None) -> bool:
    ignored_names = ignored_names or set()

    if not path.exists():
        return False

    return any(item.name not in ignored_names for item in path.iterdir())


def assert_safe_to_clear(path: Path):
    resolved = path.expanduser().resolve()

    forbidden_paths = {
        Path("/").resolve(),
        Path.home().resolve(),
        WORKSPACE_ROOT.resolve(),
        RUNTIME_ROOT.resolve(),
    }

    if resolved in forbidden_paths or len(resolved.parts) < 3:
        raise RuntimeError(f"Path terlalu berisiko untuk dibersihkan: {resolved}")


def clear_directory(path: Path):
    assert_safe_to_clear(path)

    if not path.exists():
        return

    for item in path.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()


def require_existing_file(path: Path, name: str):
    if not path.exists() or not path.is_file():
        raise FileNotFoundError(f"{name} tidak ditemukan: {path}")


def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def numeric_key(path: Path):
    nums = re.findall(r"\d+", path.stem)
    return int(nums[-1]) if nums else path.stem


def list_image_files(folder: Path):
    return sorted(
        [
            p for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        ],
        key=numeric_key,
    )


def ensure_clip_dir(df: pd.DataFrame, dataset_root: Path):
    df = df.copy()

    if "clip_dir_rel" in df.columns:
        df["clip_dir"] = df["clip_dir_rel"].apply(lambda x: str(dataset_root / str(x)))
        return df

    if "clip_dir" in df.columns:
        return df

    if "clip_dir_abs" in df.columns:
        df["clip_dir"] = df["clip_dir_abs"].astype(str)
        return df

    raise ValueError(
        "File split tidak memiliki kolom clip_dir_rel, clip_dir, atau clip_dir_abs."
    )


def sanitize_name(text: str) -> str:
    text = str(text)
    for ch in [" ", "/", "\\", ":", ",", "(", ")", "[", "]"]:
        text = text.replace(ch, "_")
    while "__" in text:
        text = text.replace("__", "_")
    return text.strip("_")


/opt/venv/lib/python3.10/site-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


### 5. Kaggle Dataset & VideoMAE Base Model Setup

In [ ]:
def setup_kaggle_credentials():
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)

    kaggle_json_path = kaggle_dir / "kaggle.json"

    if os.environ.get("KAGGLE_JSON"):
        creds = json.loads(os.environ["KAGGLE_JSON"])
        kaggle_json_path.write_text(json.dumps(creds), encoding="utf-8")
        print("Credential Kaggle dibuat dari KAGGLE_JSON.")

    elif os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        creds = {
            "username": os.environ["KAGGLE_USERNAME"],
            "key": os.environ["KAGGLE_KEY"],
        }
        kaggle_json_path.write_text(json.dumps(creds), encoding="utf-8")
        print("Credential Kaggle dibuat dari KAGGLE_USERNAME dan KAGGLE_KEY.")

    elif (WORKSPACE_ROOT / "kaggle.json").exists():
        shutil.copy2(WORKSPACE_ROOT / "kaggle.json", kaggle_json_path)
        print("Credential Kaggle disalin dari WORKSPACE_ROOT/kaggle.json.")

    elif kaggle_json_path.exists():
        print("Credential Kaggle sudah tersedia.")

    else:
        raise FileNotFoundError(
            "kaggle.json tidak ditemukan. Letakkan file pada "
            f"{WORKSPACE_ROOT / 'kaggle.json'} atau {kaggle_json_path}."
        )

    os.chmod(kaggle_json_path, 0o600)


def is_valid_zip(zip_path: Path) -> bool:
    if not zip_path.exists() or zip_path.stat().st_size == 0:
        return False

    try:
        with zipfile.ZipFile(zip_path, "r") as zip_file:
            bad_file = zip_file.testzip()
            return bad_file is None
    except zipfile.BadZipFile:
        return False


def download_kaggle_dataset() -> list[Path]:
    print_section("DOWNLOAD DATASET DARI KAGGLE")

    KAGGLE_ZIP_DIR.mkdir(parents=True, exist_ok=True)

    zip_files = sorted(KAGGLE_ZIP_DIR.glob("*.zip"))
    valid_zip_files = [zip_file for zip_file in zip_files if is_valid_zip(zip_file)]

    if valid_zip_files and not FORCE_REDOWNLOAD_DATASET:
        print("ZIP dataset valid sudah tersedia. Download dilewati.")
        return valid_zip_files

    setup_kaggle_credentials()

    for zip_file in zip_files:
        zip_file.unlink()

    run_command([
        "kaggle",
        "datasets",
        "download",
        "-d",
        KAGGLE_DATASET_SLUG,
        "-p",
        str(KAGGLE_ZIP_DIR),
    ])

    zip_files = sorted(KAGGLE_ZIP_DIR.glob("*.zip"))

    if not zip_files:
        raise FileNotFoundError("Download selesai, tetapi file ZIP tidak ditemukan.")

    invalid_zip_files = [zip_file for zip_file in zip_files if not is_valid_zip(zip_file)]

    if invalid_zip_files:
        raise RuntimeError(
            "Terdapat ZIP dataset yang tidak valid: "
            f"{[path.name for path in invalid_zip_files]}"
        )

    return zip_files


def extract_dataset(zip_files: list[Path]):
    print_section("EKSTRAKSI DATASET")

    DATASET_ROOT.mkdir(parents=True, exist_ok=True)

    extract_marker = DATASET_ROOT / ".extract_complete.json"

    dataset_has_content = has_directory_content(
        DATASET_ROOT,
        ignored_names={".extract_complete.json"},
    )

    if extract_marker.exists() and dataset_has_content and not FORCE_REEXTRACT_DATASET:
        print("Dataset sudah diekstrak. Ekstraksi dilewati.")
        return

    if dataset_has_content and not FORCE_REEXTRACT_DATASET:
        print("DATASET_ROOT sudah berisi data. Ekstraksi dilewati.")
        print_kv("DATASET_ROOT", DATASET_ROOT)
        return

    if FORCE_REEXTRACT_DATASET:
        print("FORCE_REEXTRACT_DATASET=True. DATASET_ROOT akan dibersihkan.")
        clear_directory(DATASET_ROOT)

    DATASET_ROOT.mkdir(parents=True, exist_ok=True)

    for zip_file in zip_files:
        print(f"Extracting {zip_file.name} -> {DATASET_ROOT}")

        with zipfile.ZipFile(zip_file, "r") as zf:
            bad_file = zf.testzip()

            if bad_file is not None:
                raise RuntimeError(f"ZIP tidak valid. File bermasalah: {bad_file}")

            zf.extractall(DATASET_ROOT)

    marker_payload = {
        "dataset_slug": KAGGLE_DATASET_SLUG,
        "dataset_root": str(DATASET_ROOT),
        "zip_files": [str(zip_file) for zip_file in zip_files],
        "completed_at": datetime.utcnow().isoformat() + "Z",
    }

    extract_marker.write_text(
        json.dumps(marker_payload, indent=2),
        encoding="utf-8",
    )

    print("Ekstraksi dataset selesai.")


def setup_dataset():
    print_section("SETUP DATASET")

    RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)

    dataset_ready = has_directory_content(
        DATASET_ROOT,
        ignored_names={".extract_complete.json"},
    )

    if dataset_ready and not FORCE_REEXTRACT_DATASET:
        print("Dataset sudah tersedia. Download dan ekstraksi dilewati.")
        print_kv("DATASET_ROOT", DATASET_ROOT)
        return

    if not AUTO_DOWNLOAD_DATASET:
        raise FileNotFoundError(
            "Dataset belum tersedia dan AUTO_DOWNLOAD_DATASET=False.\n"
            f"DATASET_ROOT: {DATASET_ROOT}"
        )

    zip_files = download_kaggle_dataset()
    extract_dataset(zip_files)


def has_hf_checkpoint_files(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False

    has_config = (path / "config.json").exists()
    has_weight = any([
        (path / "model.safetensors").exists(),
        (path / "pytorch_model.bin").exists(),
        (path / "tf_model.h5").exists(),
    ])

    return has_config and has_weight


def setup_videomae_base_checkpoint():
    print_section("SETUP CHECKPOINT BASE VIDEOMAE")

    if has_hf_checkpoint_files(MODEL_CHECKPOINT) and not FORCE_REDOWNLOAD_VIDEOMAE_BASE_MODEL:
        print("Checkpoint base VideoMAE sudah tersedia. Download dilewati.")
        print_kv("MODEL_CHECKPOINT", MODEL_CHECKPOINT)
        return

    if not AUTO_DOWNLOAD_VIDEOMAE_BASE_MODEL:
        raise FileNotFoundError(
            "Checkpoint base VideoMAE belum tersedia dan AUTO_DOWNLOAD_VIDEOMAE_BASE_MODEL=False.\n"
            f"MODEL_CHECKPOINT: {MODEL_CHECKPOINT}"
        )

    setup_kaggle_credentials()
    MODEL_ROOT.mkdir(parents=True, exist_ok=True)

    if FORCE_REDOWNLOAD_VIDEOMAE_BASE_MODEL:
        clear_directory(MODEL_ROOT)

    run_command([
        "kaggle",
        "models",
        "instances",
        "versions",
        "download",
        KAGGLE_MODEL_VERSION_SLUG,
        "-p",
        str(MODEL_ROOT),
        "--untar",
    ])

    if not has_hf_checkpoint_files(MODEL_CHECKPOINT):
        raise FileNotFoundError(
            "Download checkpoint base VideoMAE selesai, tetapi file HuggingFace "
            "tidak ditemukan. Periksa MODEL_CHECKPOINT."
        )


### 6. Load Index, Experiment Configs, and Manifest

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    code: str
    description: str
    num_frames: int
    stride: int
    final_image_size: int
    downscale_first: int | None = None


def validate_required_index_files():
    required_files = [
        "dataset_info.json",
        "experiment_configs.json",
        "sampling_preview.csv",
        "train_split.csv",
        "val_split.csv",
        "test_split.csv",
    ]

    for filename in required_files:
        require_existing_file(INDEX_ROOT / filename, filename)


def load_experiment_configs() -> dict[str, ExperimentConfig]:
    experiment_configs_dict = load_json(INDEX_ROOT / "experiment_configs.json")
    return {
        key: ExperimentConfig(**value)
        for key, value in experiment_configs_dict.items()
    }


def load_test_dataframe() -> tuple[pd.DataFrame, int, pd.DataFrame]:
    dataset_info = load_json(INDEX_ROOT / "dataset_info.json")
    source_clip_len = int(dataset_info["source_clip_len"])

    test_df = pd.read_csv(INDEX_ROOT / "test_split.csv")
    sampling_preview_df = pd.read_csv(INDEX_ROOT / "sampling_preview.csv")

    required_cols = {
        "label",
        "label_name",
        "clip_dir_rel",
        "num_frames",
        "group_id",
    }

    missing_cols = required_cols - set(test_df.columns)

    if missing_cols:
        raise ValueError(f"test_split.csv tidak memiliki kolom wajib: {sorted(missing_cols)}")

    test_df = ensure_clip_dir(test_df, DATASET_ROOT)

    invalid = test_df[test_df["num_frames"].astype(int) != source_clip_len]

    if len(invalid) > 0:
        raise ValueError(
            f"Ditemukan {len(invalid)} klip test dengan num_frames != {source_clip_len}."
        )

    return test_df, source_clip_len, sampling_preview_df


def read_inference_manifest() -> pd.DataFrame:
    require_existing_file(INFERENCE_MANIFEST_PATH, "inference_manifest.csv")

    manifest_df = pd.read_csv(INFERENCE_MANIFEST_PATH)
    required_cols = {
        "enabled",
        "experiment_code",
        "model_kind",
        "artifact_version",
        "study_code",
        "study_name",
    }

    missing_cols = required_cols - set(manifest_df.columns)

    if missing_cols:
        raise ValueError(f"Kolom inference_manifest.csv belum lengkap: {sorted(missing_cols)}")

    manifest_df = manifest_df[manifest_df["enabled"].astype(int) == 1].copy()
    manifest_df = manifest_df.reset_index(drop=True)

    valid_model_kinds = {"spatial_vit", "spatiotemporal_videomae"}
    invalid_model_kinds = set(manifest_df["model_kind"]) - valid_model_kinds

    if invalid_model_kinds:
        raise ValueError(f"model_kind tidak valid: {sorted(invalid_model_kinds)}")

    return manifest_df


def get_experiment_config(experiment_configs: dict[str, ExperimentConfig], code: str) -> ExperimentConfig:
    if code not in experiment_configs:
        raise ValueError(f"Kode eksperimen {code} tidak ada pada experiment_configs.json.")

    return experiment_configs[code]


### 7. Temporal Sampling, Transform, and Dataset

In [ ]:
def compute_temporal_sampling_plan(
    total_frames: int,
    num_frames: int,
    stride: int,
):
    if total_frames <= 0:
        raise ValueError("total_frames harus > 0")

    if num_frames <= 0:
        raise ValueError("num_frames harus > 0")

    if stride <= 0:
        raise ValueError("stride harus > 0")

    required_length = (num_frames - 1) * stride + 1

    if required_length > total_frames:
        raise ValueError(
            f"Konfigurasi tidak valid: T={num_frames}, stride={stride}, "
            f"membutuhkan L={required_length}, tetapi klip hanya memiliki N={total_frames}."
        )

    remaining = total_frames - required_length
    discard_left = remaining // 2
    discard_right = remaining - discard_left

    start = discard_left
    sampled_indices = [start + k * stride for k in range(num_frames)]

    return {
        "N": total_frames,
        "T": num_frames,
        "tau": stride,
        "L": required_length,
        "R": remaining,
        "discard_left": discard_left,
        "discard_right": discard_right,
        "sampled_indices": sampled_indices,
    }


def sample_frame_paths_from_clip(
    frame_files: list[Path],
    num_frames: int,
    stride: int,
):
    plan = compute_temporal_sampling_plan(
        total_frames=len(frame_files),
        num_frames=num_frames,
        stride=stride,
    )

    selected = [frame_files[i] for i in plan["sampled_indices"]]
    return selected, plan


class ClipTransform:
    """
    Transform untuk inference/testing.
    Tidak ada augmentasi; hanya resize sesuai konfigurasi eksperimen dan normalisasi.
    """

    def __init__(
        self,
        image_size: int,
        downscale_first: int | None = None,
        mean: tuple[float, float, float] = NORM_MEAN,
        std: tuple[float, float, float] = NORM_STD,
    ):
        self.image_size = image_size
        self.downscale_first = downscale_first
        self.mean = mean
        self.std = std

    def _resize_for_experiment(self, img: Image.Image):
        img = img.convert("RGB")

        if self.downscale_first is not None:
            img = TF.resize(
                img,
                size=[self.downscale_first, self.downscale_first],
                interpolation=InterpolationMode.BILINEAR,
            )
            img = TF.resize(
                img,
                size=[self.image_size, self.image_size],
                interpolation=InterpolationMode.BILINEAR,
            )
        else:
            img = TF.resize(
                img,
                size=[self.image_size, self.image_size],
                interpolation=InterpolationMode.BILINEAR,
            )

        return img

    def __call__(self, pil_images: list[Image.Image]):
        tensors = []

        for img in pil_images:
            img = self._resize_for_experiment(img)
            tensor = TF.to_tensor(img)
            tensor = TF.normalize(tensor, mean=self.mean, std=self.std)
            tensors.append(tensor)

        return torch.stack(tensors, dim=0)  # [T, C, H, W]


class ClipDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        experiment_cfg: ExperimentConfig,
        source_clip_len: int,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.cfg = experiment_cfg
        self.source_clip_len = int(source_clip_len)

        self.transform = ClipTransform(
            image_size=self.cfg.final_image_size,
            downscale_first=self.cfg.downscale_first,
            mean=NORM_MEAN,
            std=NORM_STD,
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        clip_dir = Path(row["clip_dir"])
        label = int(row["label"])

        frame_files = list_image_files(clip_dir)

        if len(frame_files) != self.source_clip_len:
            raise ValueError(
                f"Folder {clip_dir} berisi {len(frame_files)} frame, "
                f"padahal harus {self.source_clip_len} frame."
            )

        selected_files, plan = sample_frame_paths_from_clip(
            frame_files=frame_files,
            num_frames=self.cfg.num_frames,
            stride=self.cfg.stride,
        )

        images = []
        for fp in selected_files:
            with Image.open(fp) as img:
                images.append(img.convert("RGB"))

        clip_tensor = self.transform(images)
        label_tensor = torch.tensor(label, dtype=torch.long)

        meta = {
            "clip_dir": str(clip_dir),
            "label_name": row["label_name"],
            "group_id": row["group_id"] if "group_id" in row.index else None,
            "sampled_indices": torch.tensor(plan["sampled_indices"], dtype=torch.long),
        }

        if "source_clip_name" in row.index:
            meta["source_clip_name"] = row["source_clip_name"]

        if "clip_folder_name" in row.index:
            meta["clip_folder_name"] = row["clip_folder_name"]

        return clip_tensor, label_tensor, meta


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)


def make_test_loader(dataset: Dataset, model_kind: str):
    generator = torch.Generator()
    generator.manual_seed(SEED + 33)

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
        worker_init_fn=seed_worker,
        generator=generator,
    )


### 8. W&B Artifact Derivation and Download

In [ ]:
def derive_artifact_name(experiment_code: str, model_kind: str) -> str:
    if model_kind == "spatial_vit":
        return f"{experiment_code}_spatial_vit_{SPATIAL_MODEL_NAME}_outputs"

    if model_kind == "spatiotemporal_videomae":
        return f"{experiment_code}_spatiotemporal_videomae_{VIDEOMAE_MODEL_NAME}_outputs"

    raise ValueError(f"model_kind tidak dikenali: {model_kind}")


def derive_checkpoint_filename(experiment_code: str, model_kind: str) -> str:
    if model_kind == "spatial_vit":
        return f"best_{experiment_code}_spatial_vit.pt"

    if model_kind == "spatiotemporal_videomae":
        return f"best_{experiment_code}_videomae.pt"

    raise ValueError(f"model_kind tidak dikenali: {model_kind}")


def build_artifact_uri(row: pd.Series) -> str:
    artifact_name = derive_artifact_name(
        experiment_code=row["experiment_code"],
        model_kind=row["model_kind"],
    )

    return (
        f"{WANDB_ENTITY}/{WANDB_PROJECT}/"
        f"{artifact_name}:{row['artifact_version']}"
    )


def build_run_name(row: pd.Series) -> str:
    return sanitize_name(
        f"{row['study_code']}__"
        f"{row['model_kind']}__"
        f"{row['study_name']}__"
    )


def build_output_dir(row: pd.Series) -> Path:
    output_dir = INFERENCE_OUTPUT_ROOT / build_run_name(row)
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def download_source_artifact(run, row: pd.Series) -> Path:
    artifact_uri = build_artifact_uri(row)
    checkpoint_filename = derive_checkpoint_filename(
        experiment_code=row["experiment_code"],
        model_kind=row["model_kind"],
    )

    print("Mengunduh artifact:", artifact_uri)

    artifact = run.use_artifact(
        artifact_uri,
        type="experiment-output",
    )

    artifact_dir = Path(
        artifact.download(
            root=str(
                WANDB_ARTIFACT_DOWNLOAD_ROOT
                / sanitize_name(f"{row['experiment_code']}_{row['model_kind']}_{row['artifact_version']}")
            )
        )
    )

    checkpoint_path = artifact_dir / checkpoint_filename

    if not checkpoint_path.exists():
        pt_files = sorted(artifact_dir.rglob("*.pt"))

        if len(pt_files) == 1:
            checkpoint_path = pt_files[0]
        else:
            raise FileNotFoundError(
                "Checkpoint tidak ditemukan di artifact.\n"
                f"Expected filename: {checkpoint_filename}\n"
                f"Artifact dir: {artifact_dir}\n"
                f"PT files: {[str(p) for p in pt_files]}"
            )

    print("Checkpoint artifact:", checkpoint_path)
    return checkpoint_path


### 9. Model Builders and Checkpoint Loading

In [ ]:
def build_spatial_vit_model(cfg: ExperimentConfig):
    model = timm.create_model(
        SPATIAL_MODEL_NAME,
        pretrained=False,
        num_classes=2,
        img_size=cfg.final_image_size,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.0,
    )

    return model


def build_videomae_model(cfg: ExperimentConfig):
    if cfg.num_frames % 2 != 0:
        raise ValueError(
            f"cfg.num_frames={cfg.num_frames} tidak kompatibel dengan tubelet temporal 2."
        )

    config = VideoMAEConfig.from_pretrained(
        str(MODEL_CHECKPOINT),
        local_files_only=True,
    )

    config.num_frames = cfg.num_frames
    config.image_size = cfg.final_image_size
    config.num_labels = 2
    config.id2label = {0: "real", 1: "fake"}
    config.label2id = {"real": 0, "fake": 1}

    # Dropout tidak aktif saat model.eval(), sehingga nilai ini tidak memengaruhi inference.
    config.hidden_dropout_prob = 0.0
    config.attention_probs_dropout_prob = 0.0

    model = VideoMAEForVideoClassification.from_pretrained(
        str(MODEL_CHECKPOINT),
        config=config,
        ignore_mismatched_sizes=True,
        local_files_only=True,
    )

    return model


def build_model(row: pd.Series, cfg: ExperimentConfig):
    if row["model_kind"] == "spatial_vit":
        return build_spatial_vit_model(cfg)

    if row["model_kind"] == "spatiotemporal_videomae":
        return build_videomae_model(cfg)

    raise ValueError(f"model_kind tidak dikenali: {row['model_kind']}")


def load_checkpoint_into_model(model: nn.Module, checkpoint_path: Path):
    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
        )

    state_dict = checkpoint.get("model_state_dict", checkpoint)

    model.load_state_dict(state_dict, strict=True)

    model.to(DEVICE)
    model.eval()

    return checkpoint


### 10. Inference and Metrics

In [ ]:
@torch.no_grad()
def run_inference(model, loader, model_kind: str):
    model.eval()

    criterion = nn.CrossEntropyLoss()

    running_loss = 0.0
    total_samples = 0

    all_y_true = []
    all_y_pred = []
    all_prob_real = []
    all_prob_fake = []
    all_clip_dirs = []
    all_group_ids = []

    for clips, labels, meta in tqdm(loader, desc="Inference", dynamic_ncols=True, leave=False):
        clips = clips.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=AMP_DTYPE,
            enabled=USE_AMP,
        ):
            if model_kind == "spatial_vit":
                b, t, c, h, w = clips.shape
                frames = clips.reshape(b * t, c, h, w)
                frame_logits = model(frames)
                logits = frame_logits.view(b, t, 2).mean(dim=1)

            elif model_kind == "spatiotemporal_videomae":
                outputs = model(pixel_values=clips)
                logits = outputs.logits

            else:
                raise ValueError(f"model_kind tidak dikenali: {model_kind}")

            loss = criterion(logits, labels)

        probs = torch.softmax(logits.float(), dim=1)
        preds = torch.argmax(probs, dim=1)

        batch_size = labels.size(0)
        running_loss += float(loss.item()) * batch_size
        total_samples += batch_size

        all_y_true.extend(labels.detach().cpu().numpy().astype(int).tolist())
        all_y_pred.extend(preds.detach().cpu().numpy().astype(int).tolist())
        all_prob_real.extend(probs[:, 0].detach().cpu().numpy().astype(float).tolist())
        all_prob_fake.extend(probs[:, 1].detach().cpu().numpy().astype(float).tolist())

        all_clip_dirs.extend(list(meta["clip_dir"]))

        if "group_id" in meta:
            group_meta = meta["group_id"]
            if isinstance(group_meta, torch.Tensor):
                all_group_ids.extend(group_meta.detach().cpu().numpy().tolist())
            else:
                all_group_ids.extend(list(group_meta))
        else:
            all_group_ids.extend([None] * batch_size)

    y_true = np.asarray(all_y_true, dtype=int)
    y_pred = np.asarray(all_y_pred, dtype=int)
    prob_real = np.asarray(all_prob_real, dtype=float)
    prob_fake = np.asarray(all_prob_fake, dtype=float)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    precision_fake, recall_fake, f1_fake, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=POSITIVE_CLASS_INDEX,
        zero_division=0,
    )

    try:
        roc_auc = roc_auc_score(y_true, prob_fake)
        fpr, tpr, thresholds = roc_curve(
            y_true,
            prob_fake,
            pos_label=POSITIVE_CLASS_INDEX,
        )
    except ValueError:
        roc_auc = float("nan")
        fpr = np.asarray([])
        tpr = np.asarray([])
        thresholds = np.asarray([])

    print("Jumlah threshold ROC:", len(thresholds))
    print("Contoh threshold ROC:")
    print(thresholds[:20])
    
    metrics = {
        "test_loss_ce": float(running_loss / max(total_samples, 1)),
        "test_accuracy": float(accuracy_score(y_true, y_pred)),
        "test_precision_macro": float(precision_macro),
        "test_recall_macro": float(recall_macro),
        "test_f1_macro": float(f1_macro),
        "test_precision_weighted": float(precision_weighted),
        "test_recall_weighted": float(recall_weighted),
        "test_f1_weighted": float(f1_weighted),
        "test_precision_fake": float(precision_fake),
        "test_recall_fake": float(recall_fake),
        "test_num_roc_thresholds": int(len(thresholds)),
        "test_f1_fake": float(f1_fake),
        "test_roc_auc": float(roc_auc),
    }

    return {
        "metrics": metrics,
        "y_true": y_true,
        "y_pred": y_pred,
        "prob_real": prob_real,
        "prob_fake": prob_fake,
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "clip_dirs": all_clip_dirs,
        "group_ids": all_group_ids,
    }


### 11. Save and Log Outputs

In [ ]:
def save_outputs(result: dict, row: pd.Series, cfg: ExperimentConfig, checkpoint: dict, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)

    metrics = result["metrics"]

    metadata = {
        "experiment_code": cfg.code,
        "experiment_description": cfg.description,
        "model_kind": row["model_kind"],
        "study_code": row["study_code"],
        "study_name": row["study_name"],
        "artifact_version": row["artifact_version"],
        "num_frames": cfg.num_frames,
        "stride": cfg.stride,
        "final_image_size": cfg.final_image_size,
        "downscale_first": cfg.downscale_first,
        "positive_class": POSITIVE_CLASS_NAME,
        "positive_class_index": POSITIVE_CLASS_INDEX,
        "checkpoint_epoch": checkpoint.get("epoch"),
        "checkpoint_selected_by": checkpoint.get("checkpoint_selected_by"),
        "checkpoint_best_val_accuracy": checkpoint.get("best_val_accuracy"),
        "checkpoint_best_val_f1_macro": checkpoint.get("best_val_f1_macro"),
    }

    metrics_payload = {**metadata, **metrics}

    metrics_json_path = output_dir / "test_metrics_with_roc_auc.json"
    with open(metrics_json_path, "w", encoding="utf-8") as f:
        json.dump(metrics_payload, f, indent=2)

    metrics_csv_path = output_dir / "test_metrics_with_roc_auc.csv"
    pd.DataFrame([metrics_payload]).to_csv(metrics_csv_path, index=False)

    predictions_df = pd.DataFrame({
        "clip_dir": result["clip_dirs"],
        "group_id": result["group_ids"],
        "y_true": result["y_true"].astype(int),
        "y_pred": result["y_pred"].astype(int),
        "y_true_label": [IDX_TO_CLASS[int(x)] for x in result["y_true"]],
        "y_pred_label": [IDX_TO_CLASS[int(x)] for x in result["y_pred"]],
        "prob_real": result["prob_real"].astype(float),
        "prob_fake": result["prob_fake"].astype(float),
    })

    predictions_df["is_correct"] = predictions_df["y_true"] == predictions_df["y_pred"]

    predictions_path = output_dir / "test_predictions_with_probabilities.csv"
    predictions_df.to_csv(predictions_path, index=False)

    report = classification_report(
        result["y_true"],
        result["y_pred"],
        labels=[0, 1],
        target_names=["real", "fake"],
        output_dict=True,
        zero_division=0,
    )

    report_path = output_dir / "test_classification_report.csv"
    pd.DataFrame(report).transpose().to_csv(report_path)

    cm = confusion_matrix(
        result["y_true"],
        result["y_pred"],
        labels=[0, 1],
    )

    cm_csv_path = output_dir / "test_confusion_matrix.csv"
    pd.DataFrame(
        cm,
        index=["actual_real", "actual_fake"],
        columns=["pred_real", "pred_fake"],
    ).to_csv(cm_csv_path)

    plt.figure(figsize=(5.5, 4.8))

    plt.imshow(
        cm,
        interpolation="nearest",
        cmap="Blues",
    )

    plt.title("Confusion Matrix")
    plt.colorbar()

    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["real", "fake"])
    plt.yticks(tick_marks, ["real", "fake"])

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")

    threshold = cm.max() / 2.0 if cm.max() > 0 else 0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text_color = "white" if cm[i, j] > threshold else "black"
            plt.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color=text_color,
                fontsize=12,
                fontweight="bold",
            )

    plt.tight_layout()

    cm_png_path = output_dir / "test_confusion_matrix.png"
    plt.savefig(cm_png_path, dpi=300, bbox_inches="tight")
    plt.close()

    roc_png_path = output_dir / "test_roc_curve.png"

    if len(result["fpr"]) > 0:
        plt.figure(figsize=(5, 5))
        plt.plot(
            result["fpr"],
            result["tpr"],
            label=f"ROC-AUC = {metrics['test_roc_auc']:.4f}",
        )
        plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve")
        plt.legend(loc="lower right")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(roc_png_path, dpi=300, bbox_inches="tight")
        plt.close()

    return {
        "metrics_json": metrics_json_path,
        "metrics_csv": metrics_csv_path,
        "predictions_csv": predictions_path,
        "classification_report_csv": report_path,
        "confusion_matrix_csv": cm_csv_path,
        "confusion_matrix_png": cm_png_path,
        "roc_curve_png": roc_png_path if roc_png_path.exists() else None,
    }


def log_outputs_to_wandb(run, row: pd.Series, cfg: ExperimentConfig, result: dict, output_paths: dict, output_dir: Path):
    metrics = result["metrics"]

    run.log({
        key.replace("test_", "test/"): value
        for key, value in metrics.items()
    })

    run.log({
        "test/confusion_matrix": wandb.Image(str(output_paths["confusion_matrix_png"]))
    })

    if output_paths["roc_curve_png"] is not None:
        run.log({
            "test/roc_curve": wandb.Image(str(output_paths["roc_curve_png"]))
        })

    run.log({
        "test/predictions": wandb.Table(
            dataframe=pd.read_csv(output_paths["predictions_csv"])
        )
    })

    output_artifact = wandb.Artifact(
        name=sanitize_name(
            f"{cfg.code}_{row['model_kind']}_{row['study_code']}_inference_outputs"
        ),
        type="inference-output",
        metadata={
            "experiment_code": cfg.code,
            "model_kind": row["model_kind"],
            "study_code": row["study_code"],
            "study_name": row["study_name"],
            "artifact_version": row["artifact_version"],
            "test_roc_auc": metrics["test_roc_auc"],
            "test_accuracy": metrics["test_accuracy"],
            "test_f1_macro": metrics["test_f1_macro"],
        },
    )

    output_artifact.add_dir(str(output_dir))
    run.log_artifact(output_artifact)


### 12. Execute All Inference Runs

In [ ]:
os.environ["WANDB_MODE"] = WANDB_MODE

setup_dataset()
setup_videomae_base_checkpoint()

validate_required_index_files()

EXPERIMENT_CONFIGS = load_experiment_configs()
test_df, SOURCE_CLIP_LEN, sampling_preview_df = load_test_dataframe()
manifest_df = read_inference_manifest()

print_section("MANIFEST")
display(manifest_df)

all_metrics_rows = []

for idx, row in manifest_df.iterrows():
    row = row.copy()
    cfg = get_experiment_config(EXPERIMENT_CONFIGS, row["experiment_code"])

    run_name = build_run_name(row)
    output_dir = build_output_dir(row)

    print_section(f"[{idx + 1}/{len(manifest_df)}] {run_name}")
    print_kv("experiment_code", cfg.code)
    print_kv("description", cfg.description)
    print_kv("model_kind", row["model_kind"])
    print_kv("artifact_version", row["artifact_version"])
    print_kv("study_code", row["study_code"])
    print_kv("study_name", row["study_name"])
    print_kv("num_frames", cfg.num_frames)
    print_kv("stride", cfg.stride)
    print_kv("final_image_size", cfg.final_image_size)

    dataset = ClipDataset(
        dataframe=test_df,
        experiment_cfg=cfg,
        source_clip_len=SOURCE_CLIP_LEN,
    )

    test_loader = make_test_loader(
        dataset=dataset,
        model_kind=row["model_kind"],
    )

    run_config = {
        **row.to_dict(),
        "wandb_group": WANDB_GROUP,
        "experiment_description": cfg.description,
        "num_frames": cfg.num_frames,
        "stride": cfg.stride,
        "final_image_size": cfg.final_image_size,
        "downscale_first": cfg.downscale_first,
        "source_clip_len": SOURCE_CLIP_LEN,
        "positive_class": POSITIVE_CLASS_NAME,
        "positive_class_index": POSITIVE_CLASS_INDEX,
        "source_artifact_uri": build_artifact_uri(row),
    }

    tags = [
        "inference",
        str(cfg.code),
        str(row["model_kind"]),
        str(row["study_code"]),
        str(row["artifact_version"]),
    ]

    if wandb.run is not None:
        wandb.finish()

    with wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=run_name,
        group=WANDB_GROUP,
        job_type=WANDB_JOB_TYPE,
        tags=tags,
        config=run_config,
        dir=str(output_dir),
        reinit=True,
    ) as run:
        checkpoint_path = download_source_artifact(run, row)

        model = build_model(row, cfg)
        checkpoint = load_checkpoint_into_model(model, checkpoint_path)

        result = run_inference(
            model=model,
            loader=test_loader,
            model_kind=row["model_kind"],
        )

        output_paths = save_outputs(
            result=result,
            row=row,
            cfg=cfg,
            checkpoint=checkpoint,
            output_dir=output_dir,
        )

        log_outputs_to_wandb(
            run=run,
            row=row,
            cfg=cfg,
            result=result,
            output_paths=output_paths,
            output_dir=output_dir,
        )

        metrics_row = {
            "run_name": run_name,
            "experiment_code": cfg.code,
            "experiment_description": cfg.description,
            "model_kind": row["model_kind"],
            "artifact_version": row["artifact_version"],
            "study_code": row["study_code"],
            "study_name": row["study_name"],
            **result["metrics"],
        }

        all_metrics_rows.append(metrics_row)

    del model, dataset, test_loader

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(all_metrics_rows)
summary_path = INFERENCE_OUTPUT_ROOT / "inference_summary_all_runs.csv"
summary_df.to_csv(summary_path, index=False)

print_section("RINGKASAN SEMUA INFERENCE")
display(summary_df)
print("Summary disimpan ke:", summary_path)


SETUP DATASET

DOWNLOAD DATASET DARI KAGGLE
Credential Kaggle disalin dari WORKSPACE_ROOT/kaggle.json.
Menjalankan: kaggle datasets download -d afenmarbun/wilddeepfake-46f -p /shared-docker/wdf46f_runtime/kaggle_zip

STDOUT:
Dataset URL: https://www.kaggle.com/datasets/afenmarbun/wilddeepfake-46f
License(s): unknown



STDERR:

  0%|          | 0.00/56.7G [00:00<?, ?B/s]
  0%|          | 201M/56.7G [00:00<00:28, 2.10GB/s]
  1%|          | 416M/56.7G [00:00<00:27, 2.18GB/s]
  1%|          | 635M/56.7G [00:00<00:26, 2.23GB/s]
  1%|▏         | 848M/56.7G [00:00<00:26, 2.23GB/s]
  2%|▏         | 1.04G/56.7G [00:00<00:26, 2.23GB/s]
  2%|▏         | 1.25G/56.7G [00:00<00:26, 2.23GB/s]
  3%|▎         | 1.45G/56.7G [00:00<00:27, 2.19GB/s]
  3%|▎         | 1.66G/56.7G [00:00<00:26, 2.20GB/s]
  3%|▎         | 1.87G/56.7G [00:00<00:26, 2.21GB/s]
  4%|▎         | 2.08G/56.7G [00:01<00:26, 2.20GB/s]
  4%|▍         | 2.29G/56.7G [00:01<00:26, 2.23GB/s]
  4%|▍         | 2.50G/56.7G [00:01<00:26, 2.2

,enabled,experiment_code,model_kind,artifact_version,study_code,study_name
0,1,E02,spatial_vit,latest,E02,Eksperimen utama 2
1,1,E02,spatiotemporal_videomae,latest,E02,Eksperimen utama 2
2,1,E03,spatial_vit,latest,E03,Eksperimen utama 3
3,1,E03,spatiotemporal_videomae,latest,E03,Eksperimen utama 3
4,1,E04,spatial_vit,latest,E04,Eksperimen utama 4
5,1,E04,spatiotemporal_videomae,latest,E04,Eksperimen utama 4
6,1,E05,spatial_vit,latest,E05,Eksperimen utama 5
7,1,E05,spatiotemporal_videomae,latest,E05,Eksperimen utama 5
8,1,E01,spatial_vit,v11,M01,Eksperimen Baseline
9,1,E01,spatiotemporal_videomae,v12,M01,Eksperimen Baseline



[1/22] E02_spatial_vit_Eksperimen_utama_2
experiment_code                 : E02
description                     : Penurunan resolusi input model menjadi 112 x 112 untuk menguji ketahanan terhadap pengurangan informasi spasial
model_kind                      : spatial_vit
artifact_version                : latest
study_code                      : E02
study_name                      : Eksperimen utama 2
num_frames                      : 16
stride                          : 3
final_image_size                : 112


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: afenmarbun (afenmarbun-institut-teknologi-sumatera) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Mengunduh artifact: afenmarbun-institut-teknologi-sumatera/Tugas Akhir/E02_spatial_vit_vit_base_patch16_224.mae_outputs:latest


...

wandb: Adding directory to artifact (/shared-docker/inference_outputs/M08_spatiotemporal_videomae_Eksperimen_Scheduler)... Done. 0.0s


test/accuracy,▁
test/f1_fake,▁
test/f1_macro,▁
test/f1_weighted,▁
test/loss_ce,▁
test/num_roc_thresholds,▁
test/precision_fake,▁
test/precision_macro,▁
test/precision_weighted,▁
test/recall_fake,▁
+3,...



RINGKASAN SEMUA INFERENCE


,run_name,experiment_code,experiment_description,model_kind,artifact_version,study_code,study_name,test_loss_ce,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_precision_weighted,test_recall_weighted,test_f1_weighted,test_precision_fake,test_recall_fake,test_num_roc_thresholds,test_f1_fake,test_roc_auc
0,E02_spatial_vit_Eksperimen_utama_2,E02,Penurunan resolusi input model menjadi 112 x 1...,spatial_vit,latest,E02,Eksperimen utama 2,1.029980,0.760520,0.738290,0.756438,0.743524,0.775915,0.760520,0.764891,0.854545,0.769051,1323,0.809547,0.846453
1,E02_spatiotemporal_videomae_Eksperimen_utama_2,E02,Penurunan resolusi input model menjadi 112 x 1...,spatiotemporal_videomae,latest,E02,Eksperimen utama 2,0.729960,0.784344,0.759421,0.767502,0.762948,0.788516,0.784344,0.785997,0.849322,0.819542,1346,0.834166,0.849744
2,E03_spatial_vit_Eksperimen_utama_3,E03,Stride lebih rapat dengan 16 frame dan stride ...,spatial_vit,latest,E03,Eksperimen utama 3,0.903843,0.810644,0.787907,0.796543,0.791729,0.814273,0.810644,0.812042,0.869376,0.840112,1196,0.854494,0.880572
3,E03_spatiotemporal_videomae_Eksperimen_utama_3,E03,Stride lebih rapat dengan 16 frame dan stride ...,spatiotemporal_videomae,latest,E03,Eksperimen utama 3,0.769725,0.829208,0.809267,0.831820,0.816709,0.841756,0.829208,0.832199,0.909654,0.823749,1280,0.864573,0.901586
4,E04_spatial_vit_Eksperimen_utama_4,E04,Jumlah frame lebih pendek dengan 8 frame dan s...,spatial_vit,latest,E04,Eksperimen utama 4,0.993514,0.784344,0.759471,0.767949,0.763141,0.788770,0.784344,0.786076,0.850000,0.818607,1326,0.834008,0.858598
5,E04_spatiotemporal_videomae_Eksperimen_utama_4,E04,Jumlah frame lebih pendek dengan 8 frame dan s...,spatiotemporal_videomae,latest,E04,Eksperimen utama 4,0.707404,0.850866,0.835002,0.867197,0.842521,0.872546,0.850866,0.854254,0.951007,0.816737,1272,0.878773,0.924600
6,E05_spatial_vit_Eksperimen_utama_5,E05,Jumlah frame lebih panjang dengan 24 frame dan...,spatial_vit,latest,E05,Eksperimen utama 5,0.866714,0.802599,0.781501,0.801874,0.788079,0.815663,0.802599,0.806032,0.887055,0.804114,1240,0.843551,0.884351
7,E05_spatiotemporal_videomae_Eksperimen_utama_5,E05,Jumlah frame lebih panjang dengan 24 frame dan...,spatiotemporal_videomae,latest,E05,Eksperimen utama 5,0.691780,0.851176,0.833781,0.863181,0.841896,0.868772,0.851176,0.854293,0.941898,0.826087,1193,0.880199,0.927430
8,M01_spatial_vit_Eksperimen_Baseline,E01,"Konfigurasi dasar dengan resolusi 224 x 224, 1...",spatial_vit,v11,M01,Eksperimen Baseline,0.932979,0.792389,0.768142,0.767540,0.767839,0.792159,0.792389,0.792272,0.842351,0.844320,1202,0.843334,0.868301
9,M01_spatiotemporal_videomae_Eksperimen_Baseline,E01,"Konfigurasi dasar dengan resolusi 224 x 224, 1...",spatiotemporal_videomae,v12,M01,Eksperimen Baseline,0.655034,0.858292,0.838998,0.855806,0.845797,0.864416,0.858292,0.860003,0.917536,0.863488,1263,0.889692,0.923058


Summary disimpan ke: /shared-docker/inference_outputs/inference_summary_all_runs.csv


### 13. Recap Inference Metrics

In [ ]:
RECAP_OUTPUT_DIR = INFERENCE_OUTPUT_ROOT / "_recap"
RECAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "summary_df" not in globals():
    summary_path = INFERENCE_OUTPUT_ROOT / "inference_summary_all_runs.csv"
    require_existing_file(summary_path, "inference_summary_all_runs.csv")
    summary_df = pd.read_csv(summary_path)

if summary_df.empty:
    raise ValueError("summary_df kosong. Tidak ada hasil inference untuk direkap.")

recap_df = summary_df.copy()

# Bersihkan kolom teks
text_cols = [
    "experiment_code",
    "experiment_description",
    "model_kind",
    "study_code",
    "study_name",
    "artifact_version",
    "run_name",
]

for col in text_cols:
    if col in recap_df.columns:
        recap_df[col] = recap_df[col].astype(str).str.strip()

# Kolom yang ingin ditampilkan pada rekap
recap_columns = [
    "study_code",
    "study_name",
    "experiment_code",
    "experiment_description",
    "model_kind",
    "artifact_version",
    "test_accuracy",
    "test_precision_fake",
    "test_recall_fake",
    "test_f1_fake",
    "test_f1_macro",
    "test_roc_auc",
    "test_loss_ce",
    "run_name",
]

recap_columns = [
    col for col in recap_columns
    if col in recap_df.columns
]

recap_df = recap_df[recap_columns].copy()

# Bulatkan nilai metrik
metric_cols = [
    "test_accuracy",
    "test_precision_fake",
    "test_recall_fake",
    "test_f1_fake",
    "test_f1_macro",
    "test_roc_auc",
    "test_loss_ce",
]

for col in metric_cols:
    if col in recap_df.columns:
        recap_df[col] = pd.to_numeric(recap_df[col], errors="coerce").round(4)

# Urutkan agar mudah dibaca
sort_cols = [
    "study_code",
    "experiment_code",
    "model_kind",
    "artifact_version",
]

sort_cols = [
    col for col in sort_cols
    if col in recap_df.columns
]

recap_df = recap_df.sort_values(sort_cols).reset_index(drop=True)

# Simpan CSV
recap_path = RECAP_OUTPUT_DIR / "simple_inference_metrics_recap.csv"
recap_df.to_csv(recap_path, index=False)

print_section("SIMPLE INFERENCE METRICS RECAP")
display(recap_df)

print("Rekap metrik disimpan ke:", recap_path)


SIMPLE INFERENCE METRICS RECAP


,study_code,study_name,experiment_code,experiment_description,model_kind,artifact_version,test_accuracy,test_precision_fake,test_recall_fake,test_f1_fake,test_f1_macro,test_roc_auc,test_loss_ce,run_name
0,E02,Eksperimen utama 2,E02,Penurunan resolusi input model menjadi 112 x 1...,spatial_vit,latest,0.7605,0.8545,0.7691,0.8095,0.7435,0.8465,1.0300,E02_spatial_vit_Eksperimen_utama_2
1,E02,Eksperimen utama 2,E02,Penurunan resolusi input model menjadi 112 x 1...,spatiotemporal_videomae,latest,0.7843,0.8493,0.8195,0.8342,0.7629,0.8497,0.7300,E02_spatiotemporal_videomae_Eksperimen_utama_2
2,E03,Eksperimen utama 3,E03,Stride lebih rapat dengan 16 frame dan stride ...,spatial_vit,latest,0.8106,0.8694,0.8401,0.8545,0.7917,0.8806,0.9038,E03_spatial_vit_Eksperimen_utama_3
3,E03,Eksperimen utama 3,E03,Stride lebih rapat dengan 16 frame dan stride ...,spatiotemporal_videomae,latest,0.8292,0.9097,0.8237,0.8646,0.8167,0.9016,0.7697,E03_spatiotemporal_videomae_Eksperimen_utama_3
4,E04,Eksperimen utama 4,E04,Jumlah frame lebih pendek dengan 8 frame dan s...,spatial_vit,latest,0.7843,0.8500,0.8186,0.8340,0.7631,0.8586,0.9935,E04_spatial_vit_Eksperimen_utama_4
5,E04,Eksperimen utama 4,E04,Jumlah frame lebih pendek dengan 8 frame dan s...,spatiotemporal_videomae,latest,0.8509,0.9510,0.8167,0.8788,0.8425,0.9246,0.7074,E04_spatiotemporal_videomae_Eksperimen_utama_4
6,E05,Eksperimen utama 5,E05,Jumlah frame lebih panjang dengan 24 frame dan...,spatial_vit,latest,0.8026,0.8871,0.8041,0.8436,0.7881,0.8844,0.8667,E05_spatial_vit_Eksperimen_utama_5
7,E05,Eksperimen utama 5,E05,Jumlah frame lebih panjang dengan 24 frame dan...,spatiotemporal_videomae,latest,0.8512,0.9419,0.8261,0.8802,0.8419,0.9274,0.6918,E05_spatiotemporal_videomae_Eksperimen_utama_5
8,M01,Eksperimen Baseline,E01,"Konfigurasi dasar dengan resolusi 224 x 224, 1...",spatial_vit,v11,0.7924,0.8424,0.8443,0.8433,0.7678,0.8683,0.9330,M01_spatial_vit_Eksperimen_Baseline
9,M01,Eksperimen Baseline,E01,"Konfigurasi dasar dengan resolusi 224 x 224, 1...",spatiotemporal_videomae,v12,0.8583,0.9175,0.8635,0.8897,0.8458,0.9231,0.6550,M01_spatiotemporal_videomae_Eksperimen_Baseline


Rekap metrik disimpan ke: /shared-docker/inference_outputs/_recap/simple_inference_metrics_recap.csv
